In [5]:
import networkx as nx
import numpy as np


# Exercise 1

## 1

In [3]:

path_txt = "ca-AstroPh.txt"

graph = nx.Graph()

with open(path_txt, "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        node1, node2 = map(int, line.split())
        if graph.has_edge(node1, node2):
            graph[node1][node2]['weight'] += 1
        else:
            graph.add_edge(node1, node2, weight=1)

print(graph.number_of_nodes())
print(graph.number_of_edges())


18772
198110


## 2

In [6]:
node_num_neighbors = {}
node_num_egonet_edges = {}
node_total_egonet_weight = {}
node_principal_eigenvalue = {}

for node in graph.nodes():

    neighbor_list = list(graph.neighbors(node))
    node_num_neighbors[node] = len(neighbor_list)

    egonet_subgraph = graph.subgraph(neighbor_list)
    node_num_egonet_edges[node] = egonet_subgraph.number_of_edges()

    weight_sum = 0
    for node1, node2, data in egonet_subgraph.edges(data=True):
        weight_sum += data.get("weight", 1)
    node_total_egonet_weight[node] = weight_sum

    if egonet_subgraph.number_of_nodes() > 0:
        nodes_in_egonet = list(egonet_subgraph.nodes())
        index_map = {n: i for i, n in enumerate(nodes_in_egonet)}
        adjacency_matrix = np.zeros((len(nodes_in_egonet), len(nodes_in_egonet)))

        for node1, node2, data in egonet_subgraph.edges(data=True):
            w = data.get("weight", 1)
            adjacency_matrix[index_map[node1], index_map[node2]] = w
            adjacency_matrix[index_map[node2], index_map[node1]] = w

        eigenvalues = np.linalg.eigvals(adjacency_matrix)
        node_principal_eigenvalue[node] = float(max(eigenvalues.real))
    else:
        node_principal_eigenvalue[node] = 0.0

nx.set_node_attributes(graph, node_num_neighbors, "Ni")
nx.set_node_attributes(graph, node_num_egonet_edges, "Ei")
nx.set_node_attributes(graph, node_total_egonet_weight, "Wi")
nx.set_node_attributes(graph, node_principal_eigenvalue, "Lambda")
